# Check your robot dataset before submission

Practise reading a Kalanos audit and writing a short evidence note for your assignment.
By default the notebook shows three saved worked examples; set `SOURCE` below to audit your own dataset or load your own report.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**. Use an approved local Jupyter environment for private course data.

Follow your course instructions for the final submission. The audit checks recorded evidence under a named scope. Task success and policy performance require separate evaluation.

## Install

Kalanos and two helpers, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts "kalanos[numeric,hf,video]==0.10.2" huggingface-hub==1.33.0 plotly==7.1.0

## Choose your input

`SOURCE` decides what the notebook shows:

- empty: the three worked examples;
- a path ending in `.json`: a saved Kalanos report in the runtime;
- anything else: a dataset page URL, `hf://datasets/org/name@FULL_COMMIT` or a local LeRobot folder, which the notebook grades.

Pin the commit for a reproducible writeup. `PROFILE` applies only when the notebook grades. Check your instructor's scope before changing it; `numeric-core-v1` does not inspect camera frames.

To regrade the Northwestern pilot, paste `hf://datasets/IDEAS-Lab-Northwestern/datagen-dusty-v1-joint-5cam@ac57d863b06c294007bd74c9ae3046e26c2ccf0c` and keep `numeric-core-v1`.
The dataset card lists 1,040 LeRobot v2.1 episodes, five cameras and 30 fps, and the repository is 9.63 GB including video.
Grading takes about 35 minutes, and loading its 486 MB report needs about 8 GB of memory, more than a free Colab runtime reliably has.

In [ ]:
# @title Choose your input
SOURCE = ""  # @param {type:"string"}
PROFILE = "numeric-core-v1"  # @param ["numeric-core-v1", "vision-imitation-v1"]

## Load the audits

The worked examples are summaries this notebook took from Kalanos 0.10.2 reports.
Northwestern holds counts and the per-episode values of its report-only chart only, because its full report is about 486 MB.
Your own grade uses the same version.

In [ ]:
# @title Load the worked examples or your own audit
import json
import logging
import urllib.request
from collections import Counter
from pathlib import Path


# The commit in the URL pins the file, so every run shows the same examples.
EXAMPLES_URL = "https://raw.githubusercontent.com/KalanosAI/kalanos/7b69c0b5d00be31be6c37ad213c7176e491b98af/notebooks/reports/assignment_worked_examples.json"


# Statuses a graded value can carry; other statuses have no place on a band.
GRADED = ("good", "warning", "critical")


def report_only_values(report: object) -> dict | None:
    """Collect every episode's value for the channel and check with the most report-only findings.

    Parameters
    ----------
    report
        The result of `kalanos.load_report`.

    Returns
    -------
    dict or None
        The check, where it was measured, Kalanos's default bands and the review consequence, and one value per measured episode; `None` when the report has no channel-level report-only finding.
    """
    # Outside `kalanos.__all__`: the install cell pins the version this call was written for.
    from kalanos.analysis.scoring.score import consequence_for

    groups = Counter(
        (f.metric_id, f.source_field, f.stream, f.channel)
        for f in report.findings
        if f.consequence.value == "report_only" and f.channel is not None
    )
    if not groups:
        return None
    # The largest group; a tie goes to the first key in sort order.
    key, count = min(
        groups.items(),
        key=lambda item: (-item[1], tuple(part or "" for part in item[0])),
    )
    metric, source_field, stream, channel = key
    unit = next(
        f.unit
        for f in report.findings
        if (f.metric_id, f.source_field, f.stream, f.channel) == key
    )
    bare = metric.partition(".")[2]

    episodes = []
    # Values Kalanos graded from the band alone, which the guard below can check.
    banded = []
    for ep in report.episodes:
        for graded in ep.streams:
            if graded.source_field != source_field or graded.taxonomy_type != stream:
                continue
            for ch in graded.channels:
                result = ch.metrics.get(bare) if ch.channel.name == channel else None
                if result is None or result.value is None:
                    continue
                if result.status.value not in GRADED:
                    continue
                episodes.append(
                    {
                        "episode": ep.id,
                        "value": result.value,
                        "status": result.status.value,
                        "n_scored": result.evidence.get("n_scored"),
                    }
                )
                # Kalanos caps a channel that never changed at a warning, whatever its value.
                if result.evidence.get("never_changed") is not True:
                    banded.append(episodes[-1])

    policy = kalanos.load_policy(None)
    entry = policy.entry_for(bare)
    good = bad = bad_consequence = None
    higher_is_better = False
    if entry is not None:
        family, metric_policy = entry
        higher_is_better = metric_policy.higher_is_better
        band = metric_policy.thresholds.get(
            stream, metric_policy.thresholds.get("default")
        )
        if band is not None:
            good, bad = band.good, band.bad
        bad_consequence = consequence_for(
            kalanos.Severity.CRITICAL,
            metric_policy=metric_policy,
            policy=policy,
            key=f"{family}.{bare}",
        )[0].value

    def expected(value: float) -> str:
        """Return the status Kalanos's interpolation gives a value: good first, then critical."""
        if (value >= good) if higher_is_better else (value <= good):
            return "good"
        if (value <= bad) if higher_is_better else (value >= bad):
            return "critical"
        return "warning"

    # A status the bands can't reproduce means the report was graded under another policy.
    if (
        good is not None
        and bad is not None
        and any(e["status"] != expected(e["value"]) for e in banded)
    ):
        good = bad = bad_consequence = None

    return {
        "metric": metric,
        "source": source_field,
        "stream": stream,
        "channel": channel,
        "unit": unit,
        "findings": count,
        "higher_is_better": higher_is_better,
        "good": good,
        "bad": bad,
        "bad_consequence": bad_consequence,
        "episodes": episodes,
    }


def summarize(report: object) -> dict:
    """Project a Kalanos report onto the few fields this notebook charts.

    The returned dict has the same shape as the worked examples,
    so the chart and review cells treat saved and fresh audits identically.

    Parameters
    ----------
    report
        The result of `kalanos.load_report`.

    Returns
    -------
    dict
        Scope, decision counts, readiness, episode decisions, the findings that changed one, and the values behind the most frequent report-only finding.

    Raises
    ------
    ValueError
        If the result is not a Kalanos audit with recorded episode decisions.
    """
    if not isinstance(report, kalanos.Report) or report.eligibility_counts is None:
        raise ValueError("Expected a Kalanos audit with recorded episode decisions.")
    totals = report.eligibility_counts
    return {
        "version": report.producer.version if report.producer else None,
        "profile": report.scope.requirements_id if report.scope else None,
        "tier": report.scope.tier.value if report.scope else None,
        # `pass_` because `pass` is a Python keyword.
        "counts": dict(
            pass_=totals.pass_count,
            review=totals.review,
            blocked=totals.blocked,
            unknown=totals.unknown,
        ),
        "total": totals.total,
        "complete": totals.inventory_complete,
        "readiness": report.readiness.score if report.readiness else None,
        "readiness_reasons": report.readiness.reasons if report.readiness else [],
        "episodes": [
            {
                "id": ep.id,
                "status": ep.eligibility.status.value,
                "reasons": [r.detail or r.id for r in ep.eligibility.reasons],
            }
            for ep in report.episodes
            if ep.eligibility is not None
        ],
        # Only findings that changed an episode decision.
        "findings": [
            {
                "episode": f.episode_id,
                "metric": f.metric_id,
                "source": f.source_field,
                "channel": f.channel,
                "value": f.value,
                "unit": f.unit,
                "consequence": f.consequence.value,
                "camera_median": f.evidence.get("camera_median"),
            }
            for f in report.findings
            if f.consequence.value in ("review", "block")
        ],
        "report_only": report_only_values(report),
    }


source = SOURCE.strip()
native_report = None  # path of the audit JSON, when one is loaded
# Like the CLI's `--verbosity error`: hides warnings about file formats a dataset doesn't use.
logging.getLogger("kalanos").setLevel(logging.ERROR)

if not source:
    urllib.request.urlretrieve(EXAMPLES_URL, "assignment_worked_examples.json")
    audits = json.loads(Path("assignment_worked_examples.json").read_text())
else:
    import kalanos

    if source.lower().endswith(".json"):
        native_report = Path(source).expanduser()
        label = native_report.name
        report = kalanos.load_report(native_report)
    else:
        tier = "full" if PROFILE == "vision-imitation-v1" else "standard"
        # Graded straight into `report`, so a large dataset's report sits in memory once.
        report = kalanos.grade(source, bundle=PROFILE, tier=kalanos.ExecutionTier(tier))
        native_report = Path("own-report.json")
        native_report.write_text(report.model_dump_json())
        label = source
        if report.operational_errors:
            print(
                f"{len(report.operational_errors)} operational errors during the grade;"
                f" inspect {native_report}."
            )

    audits = {label: summarize(report)}
    counts = audits[label]["counts"]
    print(
        f"{label}: {audits[label]['total']} episodes, {counts['pass_']} pass,"
        f" {counts['review']} review, {counts['blocked']} blocked,"
        f" {counts['unknown']} unknown"
    )
    if source.startswith("hf://datasets/") and "@" in source:
        audits[label]["revision"] = source.rsplit("@", 1)[1]

using_saved_examples = not source

## Episode decisions

For each dataset, the notebook first checks that the decision counts add up to the known episodes and agree with the episode rows. Then it charts how each dataset's episodes split by decision, and lists each audit's scope, source revision, counts and readiness.

In [ ]:
# @title Compare episode decisions
import re

import plotly.graph_objects as go
from IPython.display import Markdown, display


# Keys match the "counts" dict.
statuses = ("pass_", "review", "blocked", "unknown")
labels = ("Pass", "Review", "Blocked", "Unknown")

# Sanity checks: the chart and table must never disagree with the report they are drawn from.
unloaded_unknown = {}
for name, audit in audits.items():
    counts, total = audit["counts"], audit["total"]
    rows = Counter(episode["status"] for episode in audit["episodes"])
    if sum(counts.values()) != total:
        raise ValueError(f"Decision counts do not sum to the report inventory: {name}")
    if not audit.get("summary_only"):
        for key in statuses:
            observed = rows[key.rstrip("_")]
            # Unknown episodes may have no loaded row, so only a surplus is an error.
            if (key == "unknown" and observed > counts[key]) or (
                key != "unknown" and observed != counts[key]
            ):
                raise ValueError(f"Episode rows disagree with {key} in {name}")
    unloaded_unknown[name] = (
        0 if audit.get("summary_only") else counts["unknown"] - rows["unknown"]
    )


def short_name(name: str) -> str:
    """Return a dataset label without its Hugging Face prefix or revision, which the scope table lists."""
    for prefix in ("hf://datasets/", "https://huggingface.co/datasets/"):
        name = name.removeprefix(prefix)
    return name.split("@", 1)[0].rstrip("/")


def percent(count: int, total: int) -> str:
    """Return a share as a whole percentage that never rounds a non-zero count to 0% or 100%."""
    share = 100 * count / total
    if 0 < share < 1:
        return "<1%"
    if 99 < share < 100:
        return ">99%"
    return f"{share:.0f}%"


# One accent: grey recedes so review, blocked and unknown carry the colour.
# All four clear 3:1 on the surface, and the three hues stay apart under colour blindness.
STATUS_COLORS = dict(
    zip(statuses, ("#94938d", "#eb6834", "#a52e31", "#4a63c8"), strict=True)
)
SURFACE, GRID, INK, MUTED = "#fcfcfb", "#e1e0d9", "#2b2b29", "#6b6a65"
FONT = {
    "family": "system-ui, -apple-system, 'Segoe UI', Roboto, sans-serif",
    "size": 16,
    "color": INK,
}
# Title and subtitle start at the plot's left edge, at the top of the figure.
TITLE = {
    "x": 0,
    "xref": "paper",
    "xanchor": "left",
    "y": 1,
    "yref": "container",
    "yanchor": "top",
    "pad": {"t": 24},
}
OUTCOMES = {"review": "sent to review", "blocked": "blocked", "unknown": "unknown"}
ROW_PX, BAR = 112, 0.6

names = list(audits)
labels_y = [short_name(name) for name in names]
totals = [audits[name]["total"] for name in names]
not_passed = [
    total - audits[name]["counts"]["pass_"]
    for name, total in zip(names, totals, strict=True)
]

fig = go.Figure()
for key, status_label in zip(statuses, labels, strict=True):
    status_counts = [audits[name]["counts"][key] for name in names]
    if not any(status_counts):
        continue
    fig.add_bar(
        name=status_label,
        y=labels_y,
        x=[100 * c / t if t else 0 for c, t in zip(status_counts, totals, strict=True)],
        text=[
            percent(c, t) if c else ""
            for c, t in zip(status_counts, totals, strict=True)
        ],
        textposition="inside",
        insidetextanchor="middle",
        # Ink reads on grey and orange; the darker red and blue take the surface colour.
        textfont={"color": INK if key in ("pass_", "review") else SURFACE, "size": 16},
        customdata=list(zip(status_counts, totals, strict=True)),
        orientation="h",
        width=BAR,
        # A surface-coloured edge leaves a gap between segments.
        marker={"color": STATUS_COLORS[key], "line": {"color": SURFACE, "width": 2}},
        hovertemplate=f"{status_label}: %{{customdata[0]:,}} of %{{customdata[1]:,}} episodes<extra>%{{y}}</extra>",
    )

for name, label, total, flagged in zip(
    names, labels_y, totals, not_passed, strict=True
):
    # The dataset's name above its bar, so long names leave the bar its full width.
    fig.add_annotation(
        x=0,
        y=label,
        text=f"<b>{label}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="bottom",
        yshift=BAR * ROW_PX / 2 + 4,
    )
    # The not-passed count, at the end of the bar.
    kinds = [key for key in OUTCOMES if audits[name]["counts"][key]]
    outcome = OUTCOMES[kinds[0]] if len(kinds) == 1 else "did not pass"
    fig.add_annotation(
        x=100,
        y=label,
        text=f"<b>{flagged:,}</b> of {total:,}<br>{outcome}"
        if flagged
        else (f"all {total:,}<br>pass" if total else "no episodes"),
        showarrow=False,
        xanchor="left",
        align="left",
        xshift=14,
        font={"color": INK if flagged else MUTED},
    )

subtitle = "Each bar splits a dataset's known episodes by Kalanos decision."
differing = [
    label
    for label, keys in (
        ("profiles", ("profile", "tier")),
        ("grader versions", ("version",)),
    )
    if len({tuple(a[k] for k in keys) for a in audits.values()}) > 1
]
if differing:
    subtitle += f"<br>{' and '.join(differing).capitalize()} differ between datasets: match the scope before comparing them."
flagged_sets = sum(1 for n in not_passed if n)
if not sum(totals):
    title = "No episodes in the inventory"
elif not flagged_sets:
    title = f"All {sum(totals):,} episodes pass"
else:
    kinds = [
        key for key in OUTCOMES if any(audits[name]["counts"][key] for name in names)
    ]
    outcome = "were " + OUTCOMES[kinds[0]] if len(kinds) == 1 else "did not pass"
    title = f"{sum(not_passed):,} of {sum(totals):,} episodes {outcome}"
    if 1 < len(names) and flagged_sets < len(names):
        title += f", all in {flagged_sets} of {len(names)} datasets"

fig.update_layout(
    font=FONT,
    paper_bgcolor=SURFACE,
    plot_bgcolor=SURFACE,
    barmode="stack",
    height=200 + ROW_PX * len(names),
    margin={"t": 140, "r": 150, "l": 24, "b": 60},
    title={
        **TITLE,
        "text": title,
        "font": {"size": 24},
        "subtitle": {"text": subtitle, "font": {"color": MUTED, "size": 16}},
    },
    # Percentages too wide for their segment are hidden; hover still gives the count.
    uniformtext={"minsize": 13, "mode": "hide"},
    legend={
        "orientation": "h",
        "x": 0,
        "xanchor": "left",
        "yref": "container",
        "y": 0.02,
        "yanchor": "bottom",
        "traceorder": "normal",
    },
)
fig.update_xaxes(
    range=[0, 100], showticklabels=False, showgrid=False, zeroline=False, showline=False
)
# Datasets top to bottom in the table's order.
fig.update_yaxes(autorange="reversed", showticklabels=False, showline=False, ticks="")
fig.show()

# Readiness reasons that repeat a count or a note shown here.
REPEATED_REASON = re.compile(
    r"^\d+ episode\(s\) (require review|have unresolved required evidence)$|^inventory incomplete"
)


def revision_link(name: str, revision: str | None) -> str:
    """Return the short revision, linked to the dataset's files at the full one."""
    if not revision:
        return ""
    return f"[`{revision[:7]}`](https://huggingface.co/datasets/{short_name(name)}/tree/{revision})"


def readiness_text(readiness: float | None) -> str:
    """Return the readiness index out of 100, or why it is missing."""
    return "not established" if readiness is None else f"{readiness:.1f}"


columns = [
    "Dataset",
    "Profile",
    "Kalanos",
    "Revision",
    "Episodes",
    *labels,
    "Readiness",
]
numeric = {"Episodes", *labels, "Readiness"}
lines = [
    "| " + " | ".join(columns) + " |",
    "| " + " | ".join("---:" if c in numeric else "---" for c in columns) + " |",
]
notes = []
for name, audit in audits.items():
    counts = audit["counts"]
    cells = [
        f"**{short_name(name)}**",
        f"{audit['profile']} / {audit['tier']}",
        audit["version"] or "",
        revision_link(name, audit.get("revision")),
        f"{audit['total']:,}",
        *(f"{counts[key]:,}" for key in statuses),
        readiness_text(audit["readiness"]),
    ]
    lines.append("| " + " | ".join(cells) + " |")
    if not audit["complete"]:
        notes.append(f"**{short_name(name)}**: the inventory is incomplete.")
    if unloaded_unknown[name]:
        notes.append(
            f"**{short_name(name)}**: {unloaded_unknown[name]:,} unknown episodes are counted but not listed in the report."
        )
    notes += [
        f"**{short_name(name)}**: {reason}"
        for reason in audit["readiness_reasons"]
        if not REPEATED_REASON.match(reason)
    ]
if any(audit["readiness"] is None for audit in audits.values()):
    notes.append(
        "Readiness is established only when the inventory is complete and every episode passes or is blocked."
    )
display(Markdown("\n".join(lines) + "".join(f"\n\n{note}" for note in notes)))

## Review queue

For each episode that did not pass, one row per finding, or one row with its reasons when it has none, up to 20 rows per dataset.
Every episode that did not pass, with all its reasons, is saved to `episode-review.csv`.

In [ ]:
# @title Inspect the evidence and export a review queue
import csv

from IPython.display import Markdown, display


TABLE_ROWS = 20

# One CSV row per episode that did not pass; saved counts-only audits contribute none.
with Path("episode-review.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["dataset", "episode", "status", "reasons"])
    for name, audit in audits.items():
        for row in audit["episodes"]:
            if row["status"] != "pass":
                writer.writerow(
                    [name, row["id"], row["status"], " | ".join(row["reasons"])]
                )


def amount(value: float | None, unit: str | None) -> str:
    """Return a value with two decimals and its unit, or an empty cell."""
    return "" if value is None else f"{value:.2f} {unit or ''}".strip()


def table_text(text: str) -> str:
    """Escape the characters that would break a Markdown table cell."""
    return text.replace("|", "\\|").replace("\n", " ")


blocks = []
for name, audit in audits.items():
    if audit.get("summary_only"):
        blocks.append(
            f"**{short_name(name)}**: saved counts only. Its full report is about 486 MB; see *Choose your input* to regrade it."
        )
        continue
    if not audit["total"]:
        blocks.append(f"**{short_name(name)}**: no episodes in the inventory.")
        continue
    attention = [row for row in audit["episodes"] if row["status"] != "pass"]
    if not attention and not unloaded_unknown[name]:
        blocks.append(f"**{short_name(name)}**: every loaded episode passed.")
        continue
    by_episode = {}
    for finding in audit["findings"]:
        by_episode.setdefault(finding["episode"], []).append(finding)
    # One row per finding; an episode with no finding shows its decision reasons.
    rows = []
    for row in attention:
        episode = f"`{row['id'].rsplit('::', 1)[-1]}`"
        findings = by_episode.get(row["id"], [])
        if not findings:
            rows.append(
                {
                    "Episode": episode,
                    "Decision": row["status"],
                    "Check": table_text("; ".join(row["reasons"])),
                }
            )
        for f in findings:
            rows.append(
                {
                    "Episode": episode,
                    "Decision": row["status"],
                    "Check": f"`{f['metric']}`",
                    "Measured on": f"`{'/'.join(filter(None, [f['source'], f['channel']])) or 'episode'}`",
                    "Value": amount(f["value"], f["unit"]),
                    "Camera median": amount(f["camera_median"], f["unit"]),
                }
            )
    columns = ["Episode", "Decision", "Check", "Measured on", "Value", "Camera median"]
    # A column that repeats one value moves into the lead line; an empty one is dropped.
    # Not when the report leaves episodes out: the lead would describe episodes nobody can see.
    shared = {
        column: rows[0].get(column, "")
        for column in ("Decision", "Check", "Measured on")
        if rows
        and not unloaded_unknown[name]
        and len({r.get(column, "") for r in rows}) == 1
    }
    columns = [c for c in columns if c not in shared and any(r.get(c) for r in rows)]
    # Unknown episodes the report counts but does not list still belong in the lead's count.
    lead = f"**{short_name(name)}**: {len(attention) + unloaded_unknown[name]:,} of {audit['total']:,} episodes "
    lead += OUTCOMES.get(shared.get("Decision", ""), "did not pass")
    if shared.get("Check"):
        lead += f", {'all ' if len(rows) > 1 else ''}for {shared['Check']}"
    if shared.get("Measured on"):
        lead += f" on {shared['Measured on']}"
    if unloaded_unknown[name]:
        lead += f"; {unloaded_unknown[name]:,} unknown episodes are counted but not listed in the report"
    numeric = {"Value", "Camera median"}
    lines = [
        lead,
        "",
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---:" if c in numeric else "---" for c in columns) + " |",
        *(
            "| " + " | ".join(r.get(c, "") for c in columns) + " |"
            for r in rows[:TABLE_ROWS]
        ),
    ]
    if len(rows) > TABLE_ROWS:
        lines += ["", f"{len(rows) - TABLE_ROWS} more rows are in the CSV."]
    blocks.append("\n".join(lines))

blocks.append(
    "Every episode that did not pass, with all its reasons, is in `episode-review.csv`."
)
if native_report is not None:
    shown = (
        native_report.relative_to(Path.cwd())
        if native_report.is_relative_to(Path.cwd())
        else native_report
    )
    blocks.append(f"Audit JSON: `{shown}`")
display(Markdown("\n\n".join(blocks)))

## Chart the evidence

One row per finding that sent an episode to review or blocked it, grouped by dataset and metric, each on its own scale. Hover over a dot for where the metric was measured.
A camera finding also shows the same camera's median across the dataset, the baseline Kalanos compared the episode with.
Counts-only examples have no findings to plot.

In [ ]:
# @title Chart the evidence behind each decision
import re

from plotly.subplots import make_subplots


CONSEQUENCES = {
    "review": ("Review", STATUS_COLORS["review"]),
    "block": ("Blocked", STATUS_COLORS["blocked"]),
}
ROW_PX, PANEL_GAP_PX = 46, 150
LEFT_PX, EDGE_PX = 130, 24


def episode_label(episode_id: str) -> str:
    """Return `Episode 8` for an id ending in `episode_000008`, else the id after its dataset."""
    match = re.search(r"episode_(\d+)$", episode_id)
    return f"Episode {int(match[1])}" if match else episode_id.rsplit("::", 1)[-1]


def measured_at(finding: dict) -> str:
    """Return where the metric was measured: its source field and channel."""
    return "/".join(filter(None, [finding["source"], finding["channel"]])) or "episode"


# One panel per dataset and metric, since each metric has its own unit and scale.
panels = {}
for name, audit in audits.items():
    for finding in audit["findings"]:
        if finding["value"] is not None:
            panels.setdefault((name, finding["metric"]), []).append(finding)

if not panels:
    print(
        "None of these decisions has a measured value to chart. The review queue above lists their reasons."
    )
else:
    # A one-row panel still gets three rows of height, so its callout fits.
    sizes = [max(len(group), 3) for group in panels.values()]
    plot_px = ROW_PX * sum(sizes) + PANEL_GAP_PX * (len(panels) - 1)
    fig = make_subplots(
        rows=len(panels),
        cols=1,
        row_heights=sizes,
        vertical_spacing=PANEL_GAP_PX / plot_px if len(panels) > 1 else 0,
    )
    in_legend = set()
    for row, ((name, metric), group) in enumerate(panels.items(), start=1):
        # Lowest value at the top; each row is one finding.
        group = sorted(group, key=lambda f: f["value"])
        places = {measured_at(f) for f in group}
        rows = [
            episode_label(f["episode"])
            + ("" if len(places) == 1 else f", {measured_at(f)}")
            for f in group
        ]
        unit = group[0]["unit"] or ""
        place = f" on {places.pop()}" if len(places) == 1 else ""
        lowest, lowest_row = group[0], rows[0]
        baselines = [
            (r, f)
            for r, f in zip(rows, group, strict=True)
            if f["camera_median"] is not None
        ]

        xs = [f["value"] for f in group] + [f["camera_median"] for _, f in baselines]
        low, high = min(xs), max(xs)
        pad = 0.15 * ((high - low) or abs(high) or 1)
        x_range = [low - pad, high + pad]

        # Camera findings compare an episode with its camera's median: draw the gap.
        for r, f in baselines:
            fig.add_scatter(
                x=[f["value"], f["camera_median"]],
                y=[r, r],
                mode="lines",
                line={"color": GRID, "width": 5},
                hoverinfo="skip",
                showlegend=False,
                row=row,
                col=1,
            )
        if baselines:
            legend = "Camera median"
            fig.add_scatter(
                x=[f["camera_median"] for _, f in baselines],
                y=[r for r, _ in baselines],
                mode="markers+text",
                text=[f"median {f['camera_median']:.3g}" for _, f in baselines],
                textposition="top center",
                textfont={"color": MUTED},
                cliponaxis=False,
                marker={
                    "color": STATUS_COLORS["pass_"],
                    "size": 18,
                    "symbol": "diamond",
                    "line": {"color": SURFACE, "width": 2},
                },
                name=legend,
                showlegend=legend not in in_legend,
                hovertemplate=f"%{{y}}: camera median %{{x:.3g}} {unit}<extra></extra>",
                row=row,
                col=1,
            )
            in_legend.add(legend)

        for consequence, (legend, color) in CONSEQUENCES.items():
            hits = [
                (r, f)
                for r, f in zip(rows, group, strict=True)
                if f["consequence"] == consequence
            ]
            if not hits:
                continue
            fig.add_scatter(
                x=[f["value"] for _, f in hits],
                y=[r for r, _ in hits],
                # The highest value gets a label when the panel has more than one row.
                mode="markers+text",
                text=[
                    f"<b>{f['value']:.3g}</b>"
                    if f is group[-1] and len(group) > 1
                    else ""
                    for _, f in hits
                ],
                textposition="middle left",
                cliponaxis=False,
                customdata=[[f["episode"], measured_at(f)] for _, f in hits],
                marker={
                    "color": color,
                    "size": 18,
                    "line": {"color": SURFACE, "width": 2},
                },
                name=legend,
                showlegend=legend not in in_legend,
                hovertemplate=f"<b>%{{y}}</b><br>%{{customdata[1]}}<br>%{{x:.3g}} {unit}<extra>{legend}</extra>",
                row=row,
                col=1,
            )
            in_legend.add(legend)

        # The signal-to-noise ratio is in decibels: below 0 dB, the signal's flicker outweighs its motion.
        if metric == "integrity.snr_db" and low < 0:
            axis = "" if row == 1 else str(row)
            fig.add_shape(
                type="rect",
                xref=f"x{axis}",
                yref=f"y{axis} domain",
                x0=x_range[0],
                x1=0,
                y0=0,
                y1=1,
                fillcolor=MUTED,
                opacity=0.08,
                line_width=0,
                layer="below",
            )
            fig.add_annotation(
                text="below 0 dB, the signal's<br>flicker outweighs its motion",
                xref=f"x{axis}",
                yref=f"y{axis} domain",
                x=x_range[0],
                y=0,
                xanchor="left",
                yanchor="bottom",
                align="left",
                xshift=6,
                yshift=6,
                showarrow=False,
                font={"color": MUTED, "size": 14},
            )

        # One callout per panel, where the data says which direction is bad:
        # a camera finding furthest from its median, or the lowest signal-to-noise ratio.
        if baselines:
            focus = max(
                group,
                key=lambda f: abs(f["value"] - (f["camera_median"] or f["value"])),
            )
        elif metric == "integrity.snr_db":
            focus = lowest
        else:
            focus = None
        if focus is not None:
            focus_row = rows[group.index(focus)]
            callout = f"<b>{focus_row}</b>: {focus['value']:.3g} {unit}".rstrip()
            median = focus["camera_median"]
            if median is not None and 0 < focus["value"] < median:
                callout += (
                    f"<br>the camera's median is {median / focus['value']:.1f}× higher"
                )
            elif median is not None and 0 < median < focus["value"]:
                callout += f"<br>{focus['value'] / median:.1f}× the camera's median"
            elif focus is lowest and len(group) > 1:
                callout += "<br>the lowest in this panel"
            # Point away from the nearer edge so the box stays inside the figure.
            on_right = focus["value"] > (x_range[0] + x_range[1]) / 2
            fig.add_annotation(
                x=focus["value"],
                y=focus_row,
                text=callout,
                ax=-150 if on_right else 150,
                ay=36,
                align="left",
                xanchor="right" if on_right else "left",
                standoff=12,
                bgcolor="white",
                bordercolor=GRID,
                borderpad=8,
                arrowcolor=MUTED,
                row=row,
                col=1,
            )

        # Rows top to bottom from the lowest value; a short panel keeps its rows centred.
        spare = (sizes[row - 1] - len(rows)) / 2
        fig.update_xaxes(range=x_range, row=row, col=1)
        fig.update_yaxes(
            categoryorder="array",
            categoryarray=rows[::-1],
            range=[-0.5 - spare, len(rows) - 0.5 + spare],
            row=row,
            col=1,
        )
        # Panel heading at the plot's left edge, above this panel's own plot area.
        fig.add_annotation(
            text=f"<b>{short_name(name)}</b><br>{metric}{place}"
            + (f" ({unit})" if unit else ""),
            align="left",
            xref="paper",
            yref=f"y{'' if row == 1 else row} domain",
            x=0,
            y=1,
            xanchor="left",
            yanchor="bottom",
            yshift=12,
            # Back from the plot's edge to the figure's text edge, past the episode labels.
            xshift=EDGE_PX - LEFT_PX,
            showarrow=False,
        )

    episodes = len({f["episode"] for group in panels.values() for f in group})
    subtitle = "One row per finding that changed a decision, lowest value at the top.<br>Each panel has its own scale."
    if any(f["camera_median"] is not None for group in panels.values() for f in group):
        subtitle += " Camera findings compare an episode<br>with the same camera's median across the dataset."
    fig.update_xaxes(
        gridcolor=GRID,
        linecolor="#c3c2b7",
        ticks="",
        zeroline=True,
        zerolinecolor="#c3c2b7",
        zerolinewidth=1,
    )
    fig.update_yaxes(showline=False, ticks="", ticksuffix="   ", showgrid=False)
    fig.update_layout(
        font=FONT,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        height=plot_px + 280,
        margin={"t": 190, "b": 60, "l": LEFT_PX, "r": 40},
        title={
            **TITLE,
            # The episode labels sit left of the plot, so the text starts at the figure's edge.
            "xref": "container",
            "pad": {"t": 24, "l": EDGE_PX},
            "text": f"What sent {episodes} {'episode' if episodes == 1 else 'episodes'} to review"
            if all(
                f["consequence"] == "review" for group in panels.values() for f in group
            )
            else f"The evidence behind {episodes} {'episode' if episodes == 1 else 'episodes'} that did not pass",
            "font": {"size": 24},
            "subtitle": {"text": subtitle, "font": {"color": MUTED, "size": 16}},
        },
        # Medians are labelled on the chart, so a legend is needed only when reviews and blocks mix.
        showlegend=len(in_legend - {"Camera median"}) > 1,
        legend={
            "orientation": "h",
            "x": 0,
            "xanchor": "left",
            "yref": "container",
            "y": 0.02,
            "yanchor": "bottom",
        },
    )
    fig.show()

## Report-only warnings

A warning lowers an episode's score but not its decision.
For each dataset with report-only findings, the notebook charts the channel and check with the most of them: every measured episode against the policy's warning line and review line.

In [ ]:
# @title Chart report-only warnings
import math


# The x-axis title and the phrase "Each bar counts episodes by …" for each check.
AXES = {
    "integrity.spike_pct": (
        "sudden changes, % of the episode's samples",
        "share of samples more than 6σ from a local window",
    ),
    "integrity.snr_db": (
        "signal-to-noise ratio (dB), higher is cleaner",
        "signal-to-noise ratio",
    ),
}
LIGHT = "#c3c2b7"
# A line label goes on its preferred side unless less than this share of the x range is left there.
LABEL_ROOM = 0.25
# The bad line's label, and the line's name in a title or callout.
LINE_NAMES = {"review": ("review", "review line"), "block": ("blocked", "block line")}


def with_unit(value: float, unit: str | None, spec: str = "g") -> str:
    """Return a value with its unit: `0.1%` for a percentage, `15 dB` otherwise."""
    if not unit:
        return f"{value:{spec}}"
    return f"{value:{spec}}{'' if unit == '%' else ' '}{unit}"


def bin_size(span: float, count: int) -> float:
    """Return a round bin width giving about 2·√n bins, between 10 and 40, across `span`.

    The width is the largest of 1, 2, 2.5 or 5 × 10^k that is no wider than `span` over the bin count.
    """
    target = min(max(2 * math.sqrt(count), 10), 40)
    raw = (span or 1) / target
    power = 10 ** math.floor(math.log10(raw))
    return next(m * power for m in (5, 2.5, 2, 1) if m * power <= raw)


def bin_index(value: float, start: float, size: float, n_bins: int) -> int:
    """Return the bin a value falls in; the last bin also holds the top edge."""
    return min(math.floor((value - start) / size), n_bins - 1)


def add_callout(
    fig: go.Figure, episode: dict, top: int, x_range: tuple, text: str, ay: int
) -> None:
    """Point a boxed note at the top of the bar holding `episode`, from the side away from the nearer edge."""
    on_right = episode["value"] > sum(x_range) / 2
    fig.add_annotation(
        x=episode["value"],
        y=top,
        text=text,
        ax=-150 if on_right else 150,
        ay=ay,
        align="left",
        xanchor="right" if on_right else "left",
        standoff=4,
        bgcolor="white",
        bordercolor=GRID,
        borderpad=8,
        arrowcolor=MUTED,
    )


def samples_line(episode: dict, unit: str | None) -> str:
    """Return `<br>k of n samples` for a percentage measured over a known sample count, else nothing."""
    if unit != "%" or not episode["n_scored"]:
        return ""
    k = round(episode["value"] / 100 * episode["n_scored"])
    return f"<br>{k:,} of {episode['n_scored']:,} samples"


not_charted = []
for name, audit in audits.items():
    block = audit.get("report_only")
    if not block:
        not_charted.append(f"**{short_name(name)}**: no report-only findings.")
        continue

    episodes = block["episodes"]
    values = [e["value"] for e in episodes]
    unit, metric = block["unit"], block["metric"]
    good, bad, higher = block["good"], block["bad"], block["higher_is_better"]
    lined = good is not None and bad is not None
    consequence = block["bad_consequence"]
    critical_color = STATUS_COLORS["blocked" if consequence == "block" else "review"]
    line_label, line_name = LINE_NAMES.get(consequence, ("critical", "critical line"))
    outcome = OUTCOMES["review" if consequence == "review" else "blocked"]

    # Bins: a round width over the values and both bounds, starting at a multiple of the width.
    edges = values + [b for b in (good, bad) if b is not None]
    low, high = min(edges), max(edges)
    size = bin_size(high - low, len(values))
    start = math.floor(low / size) * size
    n_bins = math.floor((high - start) / size) + 1
    end = start + n_bins * size
    heights = Counter(bin_index(v, start, size, n_bins) for v in values)

    if lined:
        lo, hi = sorted((good, bad))
        names = {
            "good": f"{with_unit(good, unit)} or more"
            if higher
            else f"at or under {with_unit(good, unit)}",
            "warning": f"{with_unit(bad, None)} to {with_unit(good, unit)}, report-only"
            if higher
            else f"over {with_unit(good, unit)}, report-only",
            "critical": f"at or under {with_unit(bad, unit)}, {outcome}"
            if higher
            else f"{with_unit(bad, unit)} or more, {outcome}",
        }
    else:
        names = {
            "good": "good",
            "warning": "warning, report-only",
            "critical": "critical",
        }
    colors = {"good": LIGHT, "warning": MUTED, "critical": critical_color}
    # Worst to best when higher is better, best to worst otherwise, so the legend reads along the axis.
    order = (
        ("critical", "warning", "good") if higher else ("good", "warning", "critical")
    )

    fig = go.Figure()
    for status in order:
        xs = [e["value"] for e in episodes if e["status"] == status]
        if not xs:
            continue
        fig.add_histogram(
            x=xs,
            xbins={"start": start, "end": end, "size": size},
            marker={"color": colors[status], "line": {"color": SURFACE, "width": 2}},
            name=names[status],
            hovertemplate=f"%{{x}}{'' if unit == '%' else ' '}{unit or ''}: %{{y}} episodes<extra></extra>",
        )

    reached = [e for e in episodes if e["status"] == "critical"]
    warned = sum(e["status"] == "warning" for e in episodes)

    if lined:
        fig.add_vrect(
            x0=lo, x1=hi, fillcolor=MUTED, opacity=0.06, line_width=0, layer="below"
        )
        fig.add_vline(x=good, line={"color": MUTED, "width": 1, "dash": "dot"})
        fig.add_vline(x=bad, line={"color": critical_color, "width": 2})
        # The good label prefers the line's right, the bad label its left.
        good_right = (end - good) / (end - start) >= LABEL_ROOM
        bad_right = (bad - start) / (end - start) < LABEL_ROOM
        fig.add_annotation(
            x=good,
            y=1,
            yref="paper",
            xanchor="left" if good_right else "right",
            yanchor="top",
            xshift=8 if good_right else -8,
            align="left" if good_right else "right",
            showarrow=False,
            text=f"<b>{with_unit(good, unit)}</b>: past it, a warning<br>that lowers the score<br>but not the decision",
            font={"size": 14},
        )
        fig.add_annotation(
            x=bad,
            y=1,
            yref="paper",
            xanchor="left" if bad_right else "right",
            yanchor="top",
            xshift=8 if bad_right else -8,
            align="left" if bad_right else "right",
            showarrow=False,
            text=f"<b>{with_unit(bad, unit)}: {line_label}</b>"
            + ("" if reached else "<br>no episode reaches it"),
        )
        # The episode nearest the line without reaching it.
        closest = min(
            (e for e in episodes if (e["value"] > bad if higher else e["value"] < bad)),
            key=lambda e: abs(e["value"] - bad),
            default=None,
        )
        if closest is not None:
            gap = abs(closest["value"] - bad)
            side = "above" if closest["value"] > bad else "below"
            add_callout(
                fig,
                closest,
                heights[bin_index(closest["value"], start, size, n_bins)],
                (start, end),
                f"<b>{episode_label(closest['episode'])}: {with_unit(closest['value'], unit, '.2f')}</b>"
                f"<br>{with_unit(gap, unit, '.2f')} {side} the {line_name}"
                + samples_line(closest, unit),
                -120,
            )
    if not reached:
        middle = sorted(episodes, key=lambda e: e["value"])[len(episodes) // 2]
        add_callout(
            fig,
            middle,
            heights[bin_index(middle["value"], start, size, n_bins)],
            (start, end),
            f"<b>median: {with_unit(middle['value'], unit, '.2f')}</b>"
            + samples_line(middle, unit),
            0,
        )

    if reached and consequence in LINE_NAMES:
        verb = "was" if len(reached) == 1 else "were"
        title = f"{len(reached):,} {'episode' if len(reached) == 1 else 'episodes'} {verb} {outcome}, and {warned:,} more scored a warning"
    elif reached:
        title = f"{len(reached):,} episodes graded critical, and {warned:,} more scored a warning"
    else:
        title = f"{warned:,} of {len(episodes):,} episodes scored a warning"
        if lined:
            title += f", none reached the {line_name}"

    axis_title, phrase = AXES.get(metric, (f"{metric} ({unit})",) * 2)
    subtitle = (
        f"{short_name(name)}, {metric} on {block['source']}/{block['channel']}."
        f"<br>Each bar counts episodes by {phrase}"
    )
    if len(episodes) < audit["total"]:
        subtitle += f"; {len(episodes):,} of {audit['total']:,} episodes measured"
    subtitle += "."
    if not lined:
        subtitle += "<br>This report's statuses don't match Kalanos's default policy, so no policy lines are drawn."

    fig.update_layout(
        font=FONT,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        barmode="stack",
        bargap=0,
        height=570,
        margin={"t": 140, "b": 130, "l": 70, "r": 40},
        title={
            **TITLE,
            "text": title,
            "font": {"size": 22},
            "subtitle": {"text": subtitle, "font": {"color": MUTED, "size": 15}},
        },
        legend={
            "orientation": "h",
            "x": 0,
            "xanchor": "left",
            "yref": "container",
            "y": 0.03,
            "yanchor": "bottom",
            "traceorder": "normal",
        },
    )
    fig.update_xaxes(
        title=axis_title,
        range=[start, end],
        ticksuffix=("" if unit == "%" else " ") + unit if unit else "",
        gridcolor=GRID,
        linecolor=LIGHT,
        ticks="",
        zeroline=False,
    )
    fig.update_yaxes(
        title="episodes",
        range=[0, 1.3 * max(heights.values())],
        gridcolor=GRID,
        ticks="",
        zeroline=False,
    )
    fig.show()

if not_charted:
    display(Markdown("\n\n".join(not_charted)))

## Worked answers and your writeup

| Question | What the saved evidence shows |
| --- | --- |
| Which Cube to Bowl episode would you inspect? | Episode `000008`. Its gripper camera triggered a same-camera blur finding, with variance about **6.45** versus a median near **53.9**. Replay that camera alongside a typical episode and check for motion blur, focus, or an intentional change in scene. |
| What should you check in simulated ALOHA? | Eleven episodes need review. Episode `000002` has about **−5.14 dB** signal-to-noise on the left-waist action channel. Compare the command trace with a passing episode and decide whether the small motion is expected for the task. |
| What should you check in Northwestern’s Dusty Transfer data? | Compare the next-achieved joint actions with the separate commanded-action field when diagnosing a signal anomaly. The five-camera inventory comes from the dataset card. Use replay or a vision audit for camera evidence. |
| Which dataset is better for training? | These audits use different profiles. Match the scope before comparing audit results, then verify task success, coverage and held-out policy performance separately. |

For your dataset, record the source revision, grader version, profile and counts. Choose one episode and cite the metric, source and value from the report. Describe what replay or a signal trace confirms, mark the episode keep/fix/exclude/recollect with your reason, and record the before/after counts if you re-grade. If a result is unknown, identify the missing evidence. Follow the submission instructions in your course repo.

## Reproduce the worked examples

The worked examples are summaries of earlier Kalanos reports. To check where they come from, leave `SOURCE` empty and tick `REGRADE_EXAMPLES`.
The cell grades each example's pinned source with the profile and tier its summary records, summarizes the report the same way, and compares the counts, findings and report-only values with the saved example.
The charts and tables above keep showing the saved examples. The regraded summaries are saved to `regraded_worked_examples.json`.

Cube to Bowl and ALOHA take a few minutes. The Northwestern pilot takes about 35 minutes and about 8 GB of memory, more than a free Colab runtime reliably has, so it runs only when you also tick `INCLUDE_NORTHWESTERN`.
The saved examples and a regrade both use Kalanos 0.10.2. A regrade's episode ids also carry the dataset revision, so the comparison matches findings and report-only values by episode number.

In [ ]:
# @title Regrade the worked examples and compare
REGRADE_EXAMPLES = False  # @param {type:"boolean"}
INCLUDE_NORTHWESTERN = False  # @param {type:"boolean"}


def finding_keys(audit: dict) -> list[tuple]:
    """Return each finding as a comparable tuple, with the episode id reduced to its number.

    A fresh grade from `hf://` adds the dataset revision to every episode id,
    so the comparison keeps only the part after the dataset.
    """

    def rounded(value: float | None) -> float | None:
        return None if value is None else round(value, 6)

    return sorted(
        (
            f["episode"].rsplit("::", 1)[-1],
            f["metric"],
            f["source"] or "",
            f["channel"] or "",
            rounded(f["value"]),
            rounded(f["camera_median"]),
        )
        for f in audit["findings"]
    )


def report_only_key(audit: dict) -> tuple | None:
    """Return the report-only block as a comparable tuple, with episode ids reduced as in `finding_keys`."""
    block = audit.get("report_only")
    if block is None:
        return None
    return (
        *(
            block[k]
            for k in (
                "metric",
                "source",
                "stream",
                "channel",
                "good",
                "bad",
                "bad_consequence",
            )
        ),
        sorted(
            (e["episode"].rsplit("::", 1)[-1], round(e["value"], 6), e["status"])
            for e in block["episodes"]
        ),
    )


if not REGRADE_EXAMPLES:
    print(
        "Regrade off. Tick REGRADE_EXAMPLES to reproduce the worked examples from their pinned sources."
    )
elif source:
    print("Clear SOURCE to regrade the worked examples.")
else:
    import kalanos

    regraded = {}
    for name, saved in audits.items():
        # Counts-only examples are the large ones, so they run only on request.
        if saved.get("summary_only") and not INCLUDE_NORTHWESTERN:
            print(f"{name}: skipped. Tick INCLUDE_NORTHWESTERN to regrade it.")
            continue
        profile, tier = saved["profile"], saved["tier"]
        print(f"Grading {name} with {profile} / {tier}…", flush=True)
        audit = summarize(
            kalanos.grade(
                f"hf://datasets/{name}@{saved['revision']}",
                bundle=profile,
                tier=kalanos.ExecutionTier(tier),
            )
        )
        audit["revision"] = saved["revision"]
        if saved.get("summary_only"):
            # The saved example keeps counts and report-only values only, since its full report is about 486 MB.
            audit.update(summary_only=True, episodes=[], findings=[])
        differences = [
            label
            for label, same in (
                ("counts", audit["counts"] == saved["counts"]),
                ("findings", finding_keys(audit) == finding_keys(saved)),
                (
                    "report-only values",
                    report_only_key(audit) == report_only_key(saved),
                ),
            )
            if not same
        ]
        verdict = (
            "same counts, findings and report-only values as the saved example"
            if not differences
            else f"{' and '.join(differences)} differ from the saved example"
        )
        print(f"{name}: Kalanos {audit['version']}, {verdict}\n")
        regraded[name] = audit
    if regraded:
        Path("regraded_worked_examples.json").write_text(
            json.dumps(regraded, indent=2) + "\n"
        )
        print("Regraded summaries: regraded_worked_examples.json")